In [2]:
import numpy as np

# data qubits indexed 0-3, laid out as
#   0 1
#   2 3

# each stabiliser: (pauli_type, [list of data qubits it acts on])
# stored as a list of tuples to be called later 
stabilisers = [
    ('X', [0, 1, 2, 3]),   # S1: bulk face
    ('Z', [0, 2]),         # S2: left boundary
    ('Z', [1, 3]),         # S3: right boundary
]

logical_x = ('X', [0, 2])   # left edge
logical_z = ('Z', [0, 1])   # top edge

n_qubits = 4

In [3]:
# this is a 16 dimensional Hilbert space, so logical operators and Pauli operators need to be constructed to operate in this space

I2 = np.eye(2)
pauli = {'X': np.array([[0, 1], [1, 0]]),
         'Z': np.array([[1, 0], [0, -1]])}

def pauli_on(ptype, qubits, n=n_qubits):
    """Tensor-product operator: constructs dim-16 operator in our Hilbert space.
        Applies Pauli operator 'ptype' to prescribed qubits, identity elsewhere,
        as a tensor product. """
    
    op = np.array([[1.0]])
    for j in range(n):
        op = np.kron(op, pauli[ptype] if j in qubits else I2)
    return op

# e.g. logical X is defined as Pauli X acting on qubits 0 and 2, so calling the function pauli_on(logical_x) produces the tensor product
# XL = X I X I

# build the three 16x16 stabiliser matrices and the logicals
S  = [pauli_on(p, qs) for p, qs in stabilisers]
XL = pauli_on(*logical_x)
ZL = pauli_on(*logical_z)

def commute(A, B):     return np.allclose(A @ B, B @ A)
def anticommute(A, B): return np.allclose(A @ B, -B @ A)

# logical operators must commute with all stabilisers, and must not themselves be a product of stabilisers
print("stabilisers mutually commute:",
      all(commute(S[i], S[j]) for i in range(3) for j in range(i+1, 3)))
print("XL commutes with all stabilisers:", all(commute(XL, s) for s in S))
print("ZL commutes with all stabilisers:", all(commute(ZL, s) for s in S))
print("XL and ZL anticommute:", anticommute(XL, ZL))

stabilisers mutually commute: True
XL commutes with all stabilisers: True
ZL commutes with all stabilisers: True
XL and ZL anticommute: True


In [4]:
# in order to detect errors, we must use a projector to utilise the Born rule, measuring our stabilisers to determine the syndrome

Id = np.eye(2**n_qubits)

# projector onto the simultaneous +1 eigenspace of all stabilisers
Pi = Id.copy()
for s in S:
    Pi = Pi @ (Id + s) / 2

In [5]:
# continuing, to better implement this circuit we represetn states as kets

def ket(bits):
    """Computational basis state, e.g. ket([1,0,1,0]) -> the vector for |1010>."""
    v = np.zeros(2**n_qubits)
    v[int(''.join(map(str, bits)), 2)] = 1.0
    return v

# |0_L> : project the all-zeros state into the codespace, then normalise.
ket0L = Pi @ ket([0, 0, 0, 0])
ket0L = ket0L / np.linalg.norm(ket0L)

# |1_L> : apply the logical X. (X on qubits 0 and 2 flips |0000>-> |1010>, |1111>-> |0101>.)
ket1L = XL @ ket0L

def show(name, v):
    amps = {format(i, '04b'): round(a, 3) for i, a in enumerate(v.real) if abs(a) > 1e-9}
    print(name, "=", amps)

show("|0_L>", ket0L)
show("|1_L>", ket1L)

|0_L> = {'0000': 0.707, '1111': 0.707}
|1_L> = {'0101': 0.707, '1010': 0.707}


In [6]:
# ZL acting on 0L should produce +0L:

np.allclose(ZL @ ket0L, ket0L)

True

In [7]:
# ZL acting on 1L should produce -1L:
print(np.allclose(ZL @ ket1L, -ket1L))

True


In [8]:
# 0L and 1L shoudl be orthogonal
np.vdot(ket0L, ket1L)

0.0

In [29]:
# now to test the error detection capability of the code we impose a pauli X error on qubit 0
# and expect to recover the syndrome reading (1, -1, 1)

# first we introduce some arbitrary logical state we want to protect:
alpha, beta = 0.6, 0.8
psiL = alpha * ket0L + beta * ket1L

error   = pauli_on('X', [0])     # a bit-flip on data qubit d0
psi_err = error @ psiL           # the corrupted state we hand to the detector

# syndrome_of reads each stabiliser's eigenvalue straight off the state. a pauli error keeps psi an exact +/-1 eigenstate of 
# every stabiliser, so the expectation value <state|s|state> comes out cleanly as +1 or -1
def syndrome_of(state):
    return tuple(round(np.vdot(state, s @ state).real) for s in S)

In [31]:
clean = syndrome_of(psiL)        # before the error
fired = syndrome_of(psi_err)     # after the error

names   = ["S1 (XXXX)", "S2 (Z0Z2, left)", "S3 (Z1Z3, right)"]
flagged = [names[i] for i, v in enumerate(fired) if v == -1]   # which checks flipped

print("STAGE 4  syndrome (no error) :", clean)
print("         syndrome (errored)  :", fired)
print("         fired               :", flagged or "none")

STAGE 4  syndrome (no error) : (1, 1, 1)
         syndrome (errored)  : (1, -1, 1)
         fired               : ['S2 (Z0Z2, left)']
